# Using RXIMO Explainer with River Pollution Problem

This notebook demonstrates how to use the RXIMO explainer to understand the relationship between reference points and objective values in the River Pollution problem. We'll use SHAP (SHapley Additive exPlanations) values to explain how changes in reference points affect the objective values.

## 1. Import Required Libraries

First, let's import the necessary libraries and modules.

In [7]:
import numpy as np
import polars as pl
from desdeo.explanations.explainer import ShapExplainer
from desdeo.explanations.utils import generate_biased_mean_data
import pandas as pd
import plotly.express as px

from desdeo.mcdm.reference_point_method import rpm_solve_solutions

from desdeo.problem.testproblems import dtlz2, river_pollution_problem, simple_knapsack
from desdeo.tools import payoff_table_method

from desdeo.problem import (
    Problem,
    numpy_array_to_objective_dict,
    objective_dict_to_numpy_array,
)

from desdeo.api.utils.reference_data import (
    get_reference_data_as_dataframe,
    get_reference_point_symbols,
)


In [9]:
# Suppress specific warnings
import warnings

# Filter all warnings from nevergrad's parametrization module
warnings.filterwarnings('ignore', 
                       module='nevergrad.parametrization._datalayers')

# Filter scipy COBYLA warning
warnings.filterwarnings('ignore', 
                       message='COBYLA: Invalid RHOEND.*')

## 2. Load River Pollution Problem Data

We'll load the river pollution dataset and prepare it for the RXIMO explainer.

In [10]:
def sample_input_space(ideal, nadir, n_samples: int = 20) -> np.ndarray:
    ideal = np.array(list(ideal.values()))
    nadir = np.array(list(nadir.values()))
    """Generate random samples from the input space between ideal and nadir points."""
    dims = len(ideal)
    # Generate uniform random samples between 0 and 1
    samples = np.random.random((n_samples, dims))
    # Scale and translate the samples to fit between ideal and nadir
    ranges = nadir - ideal
    samples = ideal + (samples * ranges)
    return samples

# Generate sample reference points
num_samples = 20
problem = river_pollution_problem()
num_objectives = len(problem.objectives)

# Get ideal and nadir points from the problem
ideal, nadir = payoff_table_method(problem)
problem = problem.update_ideal_and_nadir(new_ideal=ideal, new_nadir=nadir)

# Generate random reference points between ideal and nadir
reference_points = sample_input_space(ideal, nadir, n_samples=num_samples)

input_symbols, output_symbols = get_reference_point_symbols(len(problem.objectives))

#create a dataframe to store results
results_df = pd.DataFrame(columns=[*input_symbols, *output_symbols])

for ref_point in reference_points:
    dict_reference_point = numpy_array_to_objective_dict(problem, ref_point)

    # Solve the achievement scalarizing function problem
    results = rpm_solve_solutions(problem, reference_point=dict_reference_point)

    solution = results[0]  # Take the first solution

    solution_objective_vector = objective_dict_to_numpy_array(
        problem, solution.optimal_objectives
    )
    reference_point_vector = objective_dict_to_numpy_array(
        problem, dict_reference_point
    )
    row = np.concatenate((reference_point_vector, solution_objective_vector))
    results_df.loc[len(results_df)] = row

print (results_df)



c:\Users\Giomara\Documents\Projects\DESDEO_webui\.venv\Lib\site-packages\scipy\_lib\pyprima\common\preproc.py:68: UserWarning: COBYLA: Invalid MAXFUN; it should be at least num_vars + 2; it is set to 4
  warn(f'{solver}: Invalid MAXFUN; it should be at least {min_maxfun_str}; it is set to {maxfun}')
c:\Users\Giomara\Documents\Projects\DESDEO_webui\.venv\Lib\site-packages\scipy\_lib\pyprima\common\preproc.py:68: UserWarning: COBYLA: Invalid MAXFUN; it should be at least num_vars + 2; it is set to 4
  warn(f'{solver}: Invalid MAXFUN; it should be at least {min_maxfun_str}; it is set to {maxfun}')
c:\Users\Giomara\Documents\Projects\DESDEO_webui\.venv\Lib\site-packages\scipy\_lib\pyprima\common\preproc.py:68: UserWarning: COBYLA: Invalid MAXFUN; it should be at least num_vars + 2; it is set to 4
  warn(f'{solver}: Invalid MAXFUN; it should be at least {min_maxfun_str}; it is set to {maxfun}')


         z_1       z_2       z_3       z_4       z_5       f_1       f_2  \
0   6.155089  2.977612  4.231292 -6.599902  0.239233  6.110735  2.996943   
1   4.834183  2.961278  2.153990 -1.685768  0.263509  5.411012  3.008715   
2   5.192356  3.375998  3.883852 -3.644692  0.069873  5.493391  3.123612   
3   4.758451  2.961477  1.125947 -4.407182  0.189172  5.768563  3.042233   
4   5.313575  3.407801  3.502160 -4.391058  0.070844  5.496201  3.138939   
5   4.958277  2.887852  7.319495 -7.701750  0.190800  5.121193  3.068302   
6   5.456096  3.301257  0.638728 -7.977987  0.121361  5.496350  3.128547   
7   4.829661  3.135340  0.965244 -1.481696  0.150860  5.701139  3.081256   
8   5.572700  3.269866  6.179886 -7.960405  0.346401  5.825860  3.317045   
9   5.455204  3.234752  2.828217 -0.090940  0.066210  5.543503  3.063891   
10  5.560051  3.334724  2.214007 -3.324393  0.316160  5.402190  3.271668   
11  6.174697  3.378332  7.028944 -7.866512  0.199706  6.045487  3.431145   
12  5.592398

c:\Users\Giomara\Documents\Projects\DESDEO_webui\.venv\Lib\site-packages\scipy\_lib\pyprima\common\preproc.py:68: UserWarning: COBYLA: Invalid MAXFUN; it should be at least num_vars + 2; it is set to 4
  warn(f'{solver}: Invalid MAXFUN; it should be at least {min_maxfun_str}; it is set to {maxfun}')


In [17]:

# Create Polars DataFrame
pl_df = pl.DataFrame(results_df)

print("Generated dataset shape:", pl_df.shape)
print("\nFirst few rows:")
print(pl_df.head())

Generated dataset shape: (20, 10)

First few rows:
shape: (5, 10)
┌──────────┬──────────┬──────────┬───────────┬───┬──────────┬──────────┬───────────┬──────────┐
│ z_1      ┆ z_2      ┆ z_3      ┆ z_4       ┆ … ┆ f_2      ┆ f_3      ┆ f_4       ┆ f_5      │
│ ---      ┆ ---      ┆ ---      ┆ ---       ┆   ┆ ---      ┆ ---      ┆ ---       ┆ ---      │
│ f64      ┆ f64      ┆ f64      ┆ f64       ┆   ┆ f64      ┆ f64      ┆ f64       ┆ f64      │
╞══════════╪══════════╪══════════╪═══════════╪═══╪══════════╪══════════╪═══════════╪══════════╡
│ 6.155089 ┆ 2.977612 ┆ 4.231292 ┆ -6.599902 ┆ … ┆ 2.996943 ┆ 5.690439 ┆ -0.678124 ┆ 0.249002 │
│ 4.834183 ┆ 2.961278 ┆ 2.15399  ┆ -1.685768 ┆ … ┆ 3.008715 ┆ 7.251848 ┆ -0.889502 ┆ 0.105607 │
│ 5.192356 ┆ 3.375998 ┆ 3.883852 ┆ -3.644692 ┆ … ┆ 3.123612 ┆ 7.191078 ┆ -1.962246 ┆ 0.222631 │
│ 4.758451 ┆ 2.961477 ┆ 1.125947 ┆ -4.407182 ┆ … ┆ 3.042233 ┆ 6.870627 ┆ -1.102575 ┆ 0.140293 │
│ 5.313575 ┆ 3.407801 ┆ 3.50216  ┆ -4.391058 ┆ … ┆ 3.138939 ┆ 7.188801

## 3. Setup RXIMO Explainer

Now we'll set up the SHAP explainer with our data. For the river pollution problem, we have:
- Input symbols (z_1, z_2, z_3, z_4): Reference points for each objective
- Output symbols (f_1, f_2, f_3, f_4): Objective function values

In [12]:
# Create the SHAP explainer
explainer = ShapExplainer(
    problem_data=pl_df,
    input_symbols=input_symbols,
    output_symbols=output_symbols
)

print("Input symbols:", input_symbols)
print("Output symbols:", output_symbols)

Input symbols: ['z_1', 'z_2', 'z_3', 'z_4', 'z_5']
Output symbols: ['f_1', 'f_2', 'f_3', 'f_4', 'f_5']


## 4. Generate Background Data

We'll generate background data for the SHAP explainer. This data represents the baseline against which our explanations will be computed.

In [38]:
print(ideal)
print(nadir)
# select a point between ideal and nadir
target_ref_point = (np.array(list(ideal.values())) + np.array(list(nadir.values()))) / 2
print("Reference point:", target_ref_point)

# Try different solvers in order of preference
solvers = ['CVXOPT', 'SCIPY', 'OSQP']  # Using more readily available solvers
background_indices = None

# Normalize the data to improve numerical stability
data = pl_df[output_symbols].to_numpy()
data_mean = np.mean(data, axis=0)
data_std = np.std(data, axis=0)
normalized_data = (data - data_mean) / (data_std + 1e-8)
normalized_target = (target_ref_point - data_mean) / (data_std + 1e-8)

for solver in solvers:
    try:
        print(f"Trying solver: {solver}")
        background_indices = generate_biased_mean_data(
            normalized_data,
            normalized_target,
            min_size=5,
            max_size=10,
            solver=solver
        )
        if background_indices is not None:
            print(f"Successfully generated background data using {solver}")
            break
    except Exception as e:
        print(f"Solver {solver} failed: {str(e)}")

# Fallback to random sampling if all solvers fail
if background_indices is None:
    print("All solvers failed, using random sampling fallback")
    distances = np.linalg.norm(normalized_data - normalized_target, axis=1)
    weights = 1 / (distances + 1e-8)
    weights = weights / np.sum(weights)
    background_indices = np.random.choice(len(pl_df), size=10, replace=False, p=weights)

print("Background indices:", background_indices)

# Convert indices to Python integers and create background dataset
background_indices = [int(i) for i in background_indices]  # Convert to Python integers
background_df = pl.DataFrame(results_df).select(pl.all()).filter(pl.arange(0, len(pl_df)).is_in(background_indices))


target_dict = {sym: val for sym, val in zip(input_symbols, target_ref_point)}
shaps = explainer.explain_input(pl.DataFrame([target_dict]))


print("SHAP values shape:", shaps)


{'f_1': 6.34, 'f_2': 3.444871794871795, 'f_3': 7.500000000000001, 'f_4': 0.0, 'f_5': 0.0}
{'f_1': 4.751, 'f_2': 2.8666051480818924, 'f_3': 0.32111111111111956, 'f_4': -9.706666666666656, 'f_5': 0.35000000000000003}
Reference point: [ 5.5455      3.15573847  3.91055556 -4.85333333  0.175     ]
Trying solver: CVXOPT
Solver CVXOPT failed: The solver CVXOPT is not installed.
Trying solver: SCIPY
Solver SCIPY failed: Either candidate conic solvers (['SCIPY']) do not support the cones output by the problem (SOC, NonNeg, Zero), or there are not enough constraints in the problem.
Trying solver: OSQP
Solver OSQP failed: Problem is mixed-integer, but candidate QP/Conic solvers ([]) are not MIP-capable.
All solvers failed, using random sampling fallback
Background indices: [10 11 13  2  8  9 17 15  6  7]
SHAP values shape: .values =
array([[[-0.03946948,  0.00626452,  0.06246306, -0.07689489,
          0.00587777],
        [-0.02163639,  0.00334978,  0.03358797, -0.04070363,
          0.00322935]

## 5. Visualize SHAP Values

Let's create a visualization to better understand the relationships between reference points and objectives.

In [76]:
# Prepare data for visualization
impacts = []
for i, output in enumerate(output_symbols):
    for j, input_sym in enumerate(input_symbols):
        impacts.append({
            'Output': output,
            'Input': input_sym,
            'Impact': shaps.values[0, i, j]
        })

# Create DataFrame for plotting
impact_df = pd.DataFrame(impacts)

# Create heatmap
fig = px.imshow(
    shaps.values[0],
    labels=dict(x="Input Reference Points", y="Output Objectives", color="SHAP Impact"),
    x=input_symbols,
    y=output_symbols,
    title="SHAP Values: Impact of Reference Points on Objectives",
    color_continuous_scale="RdBu_r"
)

fig.show()

In [73]:
# Create a parallel coordinates plot
import plotly.graph_objects as go

# Prepare data for parallel coordinates
# First, get the background data
background_data = background_df.to_pandas()

# Get ideal and nadir values as arrays for easier handling
ideal_arr = np.array(list(ideal.values()))
nadir_arr = np.array(list(nadir.values()))

# Prepare SHAP values for coloring and annotations
max_abs_shap = np.abs(shaps.values).max()
impact_colors = {
    'positive': 'red',
    'negative': 'blue',
    'neutral': 'grey'
}

# Calculate normalized SHAP values for each input-output pair
shap_impacts = {}
for i, out_sym in enumerate(output_symbols):
    shap_impacts[out_sym] = {}
    for j, in_sym in enumerate(input_symbols):
        shap_impacts[out_sym][in_sym] = shaps.values[0, i, j]

# Create the parallel coordinates plot
fig = go.Figure()

# Add background data - separate traces for inputs and outputs
dimensions = []

# Add output dimensions
for i, col in enumerate(output_symbols):
    dimensions.append(
        dict(range=[ideal_arr[i], nadir_arr[i]],
             label=col,
             values=background_data[col])
    )

# Add background data with all dimensions
fig.add_trace(go.Parcoords(
    line=dict(
        color='rgba(200, 200, 200, 0.3)',  # Light grey with transparency
        showscale=False
    ),
    dimensions=dimensions,
    name='Background Data'
))

# Create one data point with the target reference point values
target_df = pd.DataFrame([{col: target_ref_point[i] for i, col in enumerate(output_symbols)}])

fig.add_trace(go.Parcoords(
    line=dict(color='red',        
              showscale=False
),
    dimensions=[
        dict(
            range=[ideal_arr[i], nadir_arr[i]],
            label=col,
            values=[target_ref_point[i]]  # wrap in list
        )
        for i, col in enumerate(output_symbols)
    ],
    name='Target Reference Point'
))

# Update layout with improved styling
fig.update_layout(
    title={
        'text': 'Parallel Coordinates Plot: Reference Point and Background Data<br><sub>Ranges shown between ideal and nadir points</sub>',
        'y':0.95,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    showlegend=True,
    height=600,
    font=dict(
        size=12  # Increase font size
    ),
    plot_bgcolor='white',  # White background
    paper_bgcolor='white'  # White paper color
)

# Add annotations for SHAP impacts
y_pos = 1.15  # Starting position for annotations
for out_sym in output_symbols:
    impacts = [(in_sym, shap_impacts[out_sym][in_sym]) for in_sym in input_symbols]
    # Sort impacts by absolute value
    impacts.sort(key=lambda x: abs(x[1]), reverse=True)
    
    annotation_text = f"{out_sym} impacts:<br>"
    for in_sym, impact in impacts:
        if abs(impact) > max_abs_shap * 0.1:  # Only show significant impacts
            color = impact_colors['positive'] if impact > 0 else impact_colors['negative']
            annotation_text += f'<span style="color:{color}">{in_sym}: {impact:.3f}</span><br>'
    
    fig.add_annotation(
        x=1.15,
        y=y_pos,
        text=annotation_text,
        showarrow=False,
        xref='paper',
        yref='paper',
        align='left'
    )
    y_pos -= 0.25

fig.show()
print(target_ref_point)

sol = rpm_solve_solutions(problem, reference_point={sym: val for sym, val in zip(output_symbols, target_ref_point)})[0]

print("Solution for the target reference point:")
print("Input values:", sol.optimal_objectives)

[ 5.5455      3.15573847  3.91055556 -4.85333333  0.175     ]


c:\Users\Giomara\Documents\Projects\DESDEO_webui\.venv\Lib\site-packages\scipy\_lib\pyprima\common\preproc.py:68: UserWarning:

COBYLA: Invalid MAXFUN; it should be at least num_vars + 2; it is set to 4



Solution for the target reference point:
Input values: {'f_1': 5.931543159157983, 'f_2': 3.1089110850994914, 'f_3': 6.509387219019879, 'f_4': -1.693355038025591, 'f_5': 0.20334278833636532}


## 7. Interpretation

The SHAP values and visualization show:
1. How each reference point (z_1 to z_4) influences each objective (f_1 to f_4)
2. The magnitude and direction of the influence (positive or negative)
3. The relative importance of different reference points for each objective

Red colors indicate positive impact (increasing the objective value), while blue colors indicate negative impact (decreasing the objective value).